# v11 — задача Коши: марш неявных шагов (BE/CN) на CV-невязках, сеть по z на каждом шаге

**Почему так (итог v7–v10b):** минимизация невязки на отрезке с мягким швом нейтральна к неустойчивости — гладкая и осциллирующая ветви для неё равноправны (v10: pde 3–7·10⁻⁴ ниже уровня истины, ветвь гладкая; v10b: затравка не растёт ×1.0 против ×60 у FV). Тест единственности показал: при **жёстко зафиксированном состоянии_n** продолжение по чужой ветви стоит 2–105 против 10⁻³ — минимум шага единственен; марш BE-шагов из истинного состояния прошёл полуволну DWO размахом 0.12 kg/s в такт с FV.

**Конструкция шага.** Состояние = профили h_n(z), G_n(z) на центрах CV (константы). Сеть `ZNet` только по z (RFF по z → 2×tanh → 3 выхода, жёсткие BC в анзаце, инициализация = состояние_n) представляет новое состояние h_{n+1}, G_{n+1}, π_{n+1}. Лосс = CV-невязки LSFVP (масса/импульс/энергия по 128 CV, потоки на гранях, скачки локальных потерь в граничных CV), где ∂/∂t — формула схемы: BE (u_{n+1}−u_n)/Δt с правыми частями в n+1, или CN (полусумма правых частей в n и n+1). L-BFGS по весам сети → состояние_{n+1} снимается (детач) → следующий шаг. Ни швов, ни IC-штрафов, ни keepers, ни causal-весов: состояние_n входит в невязку напрямую.

Это FV-солвер, у которого систему на шаге решает L-BFGS по весам непрерывной сети, а не Ньютон по значениям в ячейках. Дискретизация по времени — от FV; представление по z и решатель — от PINN.

**Бюджет ~1 ч на весь горизонт (по запросу):** адаптивный шаг — Δt=1.0 t* на гладком участке (t<1400 s), Δt=0.1 t* дальше (28 отсчётов на период DWO); L-BFGS ≤ 60 итераций/шаг (тёплый старт от состояния_n делает шаг дешёвым). Старт — стационар замыканий при t=0 (штатный IC: G=G0*, h=−N_SPC — холодный, как в good_enough).

**Ворота (объявлены заранее):** огибающая vs FV ≤ 5 % при t<1500 s; **onset** (10× рост локального размаха vs QS-фон) 1626 s ± 15 %; **период** 5.93 s ± 15 %; невязка шага (лосс) < 10⁻³ в среднем. Амплитуда — не ворота (BE диссипативен, эталон в реверсе шаток); контроль — рост экспоненциальный.

Чекпоинт `v11_bemarch_ckpt.pt` (ресумабельно по шагам). Файлы прошлых линий не трогаются.


In [ ]:
import sys
import math
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
import torch
import torch.nn as nn

sys.path.append("/mnt/data")
from eos_iapws_spline import EOSGridSpec, IAPWSDensitySpline

In [ ]:
@dataclass(frozen=True)
class Forcing:
    time: np.ndarray
    tau: np.ndarray
    power: np.ndarray
    flow_in: np.ndarray
    flow_out: np.ndarray
    ntpc: np.ndarray
    mdot0: float

    @classmethod
    def from_excel(cls, path, t0_ramp=2.0):
        df = pd.read_excel(path)

        cols = ["Time [s]", "Flow-in [kg/s]", "Flow-out [kg/s]", "Power [W]", "NTPC"]
        df = df[cols].apply(pd.to_numeric, errors="coerce").dropna()
        df = df.sort_values("Time [s]")

        pre = df[
            (df["Time [s]"] >= 1.0)
            & (df["Time [s]"] <= 1.8)
            & (df["Power [W]"].abs() < 1e-12)
        ]

        mdot0 = float(pre["Flow-in [kg/s]"].mean())

        df = df[df["Time [s]"] >= t0_ramp].copy()
        time = df["Time [s]"].to_numpy()
        tau = time - t0_ramp

        return cls(
            time=time,
            tau=tau,
            power=df["Power [W]"].to_numpy(),
            flow_in=df["Flow-in [kg/s]"].to_numpy(),
            flow_out=df["Flow-out [kg/s]"].to_numpy(),
            ntpc=df["NTPC"].to_numpy(),
            mdot0=mdot0,
        )

    def power_torch(self, tau):
        x = tau.detach().cpu().numpy().reshape(-1)
        y = np.interp(x, self.tau, self.power)

        return torch.as_tensor(
            y,
            dtype=tau.dtype,
            device=tau.device,
        ).reshape_as(tau)

In [ ]:
eos_path = Path("iapws_density_spline_25mpa.npz")

if eos_path.exists():
    eos = IAPWSDensitySpline.load(eos_path)
else:
    spec = EOSGridSpec(
        p_min=24.5e6,
        p_max=25.5e6,
        h_min=0.75e6,
        h_max=4.20e6,
        n_p=64,
        n_h=256,
        kx=3,
        ky=3,
        smoothing=0.0,
    )

    eos = IAPWSDensitySpline.build(spec)
    eos.save(eos_path)

In [ ]:
@dataclass(frozen=True)
class MuGridSpec:
    p_ref: float = 25.0e6
    h_min: float = 0.75e6
    h_max: float = 4.20e6
    n_h: int = 512


class IAPWSViscositySpline:
    def __init__(self, spec, h_grid, mu_grid):
        self.spec = spec
        self.h_grid = np.asarray(h_grid, dtype=np.float64)
        self.mu_grid = np.asarray(mu_grid, dtype=np.float64)
        self.spline = CubicSpline(self.h_grid, self.mu_grid, bc_type="natural")

    @classmethod
    def build(cls, spec=MuGridSpec()):
        h_grid = np.linspace(spec.h_min, spec.h_max, spec.n_h)
        mu_grid = np.empty_like(h_grid)

        for i, h in enumerate(h_grid):
            mu_grid[i] = IAPWS97(P=spec.p_ref / 1e6, h=h / 1e3).mu

        return cls(spec, h_grid, mu_grid)

    def eval(self, h_abs, clip=True):
        h = np.asarray(h_abs, dtype=np.float64)

        if clip:
            h = np.clip(h, self.h_grid[0], self.h_grid[-1])

        return {
            "mu": self.spline(h),
            "mu_h": self.spline(h, 1),
        }

    def eval_torch(self, h_abs, clip=True):
        out = self.eval(
            h_abs.detach().cpu().numpy(),
            clip=clip,
        )

        mu_value = torch.as_tensor(
            out["mu"],
            device=h_abs.device,
            dtype=h_abs.dtype,
        )

        mu_h = torch.as_tensor(
            out["mu_h"],
            device=h_abs.device,
            dtype=h_abs.dtype,
        )

        mu = mu_value + mu_h * (h_abs - h_abs.detach())

        return {
            "mu": mu,
            "mu_h": mu_h,
        }


def smooth_abs(x, eps=1e-6):
    return torch.sqrt(x * x + eps * eps)


def darcy_lambda_torch(
    Re,
    roughness,
    Dh,
    model="swamee",
    Re_transition=2300.0,
    blend_width=200.0,
):
    Re = torch.clamp(Re, min=1.0)

    lam_laminar = 64.0 / Re

    if model == "blasius":
        lam_turb = 0.3164 / Re.pow(0.25)

    elif model == "swamee":
        rel = roughness / Dh
        arg = rel / 3.7 + 5.74 / Re.pow(0.9)
        lam_turb = 0.25 / torch.log10(arg).pow(2.0)

    else:
        raise ValueError("model must be 'blasius' or 'swamee'")

    s = torch.sigmoid((Re - Re_transition) / blend_width)

    return (1.0 - s) * lam_laminar + s * lam_turb

In [ ]:
mu_spline = IAPWSViscositySpline.build(
    MuGridSpec(
        p_ref=25.0e6,
        h_min=0.75e6,
        h_max=4.20e6,
        n_h=512,
    )
)

In [ ]:
@dataclass(frozen=True)
class Case:
    L: float
    Dh: float
    A: float

    p_ref: float
    p_out: float
    dp_ch: float
    p_in: float

    rho_pc: float
    h_pc: float
    Cp_pc: float
    beta_pc: float

    nspc: float
    h0_star: float

    kin: float
    kout: float
    Lambda: float
    roughness: float
    friction_model: str
    Re0: float
    lambda0: float
    Lambda_ref: float
    g: float

    mdot0: float
    G0: float
    rho0: float
    rho0_star: float
    w0: float

    G0_star: float
    dpi_star: float
    Fr: float

    roughness: float
    friction_model: str
    Re0: float
    lambda0: float
    Lambda_ref: float

    NQ_scale: float
    mass_scale: float
    mom_mid_scale: float
    mom_delta_scale: float
    energy_scale: float

    t_end_star: float
    t_kink_star: float

    @classmethod
    def build(cls, forcing, eos, nspc=2.0, alpha=0.02):
        L = 4.2672
        Dh = 8.36e-3
        A = math.pi * Dh**2 / 4

        p_ref = 25.0e6
        p_out = 25.0e6
        dp_ch = 0.12e6
        p_in = p_out + dp_ch

        rho_pc = 317.0348
        h_pc = 2152539.0
        Cp_pc = 76444.66
        beta_pc = 0.1285075

        h0_star = -nspc
        h0_abs = h_pc + (Cp_pc / beta_pc) * h0_star

        rho0 = float(np.asarray(eos.eval(p_ref, h0_abs)["rho"]))
        rho0_star = rho0 / rho_pc

        G0 = forcing.mdot0 / A
        w0 = G0 / rho0

        kin = 20.0
        kout = 20.0
        roughness = 2.5e-5
        friction_model = "swamee"   # можно заменить на "blasius"
        g = 9.81

        mu0 = IAPWS97(P=p_ref / 1e6, h=h0_abs / 1e3).mu
        Re0 = G0 * Dh / mu0

        if Re0 < 2300.0:
            lambda0 = 64.0 / Re0
        else:
            if friction_model == "blasius":
                lambda0 = 0.3164 / Re0**0.25
            elif friction_model == "swamee":
                lambda0 = 0.25 / (
                    np.log10(roughness / (3.7 * Dh) + 5.74 / Re0**0.9) ** 2
                )
            else:
                raise ValueError("friction_model must be 'blasius' or 'swamee'")

        Lambda_ref = lambda0 * L / (2.0 * Dh)

        # оставляем поле Lambda, чтобы старый код не ломался
        Lambda = Lambda_ref

        G0_star = G0 / (rho_pc * w0)
        dpi_star = dp_ch / (rho_pc * w0**2)
        Fr = w0**2 / (g * L)

        NQ = (
            beta_pc
            / Cp_pc
            * forcing.power
            / (rho_pc * w0 * A)
        )
        NQ_scale = max(float(np.max(np.abs(NQ))), 1.0)

        mass_scale = max(abs(G0_star), 1.0)
        energy_scale = NQ_scale
        mom_mid_scale = max(abs(dpi_star), 1.0)

        delta_max = (
            1.0 / math.sqrt(2.0 * math.pi)
            / (alpha * (0.5 * (1.0 + math.erf(1.0 / (alpha * math.sqrt(2.0)))) - 0.5))
        )

        local_coeff_max = Lambda + 0.5 * max(kin, kout) * delta_max
        local_loss_scale = local_coeff_max * G0_star**2 / rho0_star

        mom_delta_scale = max(abs(dpi_star), local_loss_scale, 1.0)

        t_end_star = float(forcing.tau[-1] * w0 / L)
        t_kink_star = float(10.0 * w0 / L)

        return cls(
            L=L,
            Dh=Dh,
            A=A,
            p_ref=p_ref,
            p_out=p_out,
            dp_ch=dp_ch,
            p_in=p_in,
            rho_pc=rho_pc,
            h_pc=h_pc,
            Cp_pc=Cp_pc,
            beta_pc=beta_pc,
            nspc=nspc,
            h0_star=h0_star,
            kin=kin,
            kout=kout,
            Lambda=Lambda,
            roughness=roughness,
            friction_model=friction_model,
            Re0=Re0,
            lambda0=lambda0,
            Lambda_ref=Lambda_ref,
            g=g,
            mdot0=forcing.mdot0,
            G0=G0,
            rho0=rho0,
            rho0_star=rho0_star,
            w0=w0,
            G0_star=G0_star,
            dpi_star=dpi_star,
            Fr=Fr,
            NQ_scale=NQ_scale,
            mass_scale=mass_scale,
            mom_mid_scale=mom_mid_scale,
            mom_delta_scale=mom_delta_scale,
            energy_scale=energy_scale,
            t_end_star=t_end_star,
            t_kink_star=t_kink_star,
        )

In [ ]:
path = Path("Churkin-Kout20-Nspc2.0-Transient-Water-Vertical.xlsx")

forcing = Forcing.from_excel(path)
case = Case.build(forcing, eos, nspc=2.0, alpha=0.02)

print(f"mdot0           = {case.mdot0:.8f} kg/s")
print(f"rho0            = {case.rho0:.6f} kg/m3")
print(f"rho0*           = {case.rho0_star:.6f}")
print(f"w0              = {case.w0:.6f} m/s")
print(f"G0*             = {case.G0_star:.6f}")
print(f"Delta pi*       = {case.dpi_star:.6f}")
print(f"Fr              = {case.Fr:.6e}")
print(f"NQ scale        = {case.NQ_scale:.6f}")
print(f"mass scale      = {case.mass_scale:.6f}")
print(f"mom mid scale   = {case.mom_mid_scale:.6f}")
print(f"mom delta scale = {case.mom_delta_scale:.6f}")
print(f"energy scale    = {case.energy_scale:.6f}")
print(f"T_end*          = {case.t_end_star:.3f}")
print(f"Re0             = {case.Re0:.3f}")
print(f"lambda0         = {case.lambda0:.6f}")
print(f"Lambda_ref      = {case.Lambda_ref:.6f}")
print(f"friction model  = {case.friction_model}")

In [ ]:
class SmoothPINN(nn.Module):
    def __init__(self, t_end_star, width=128, depth=6, activation="tanh"):
        super().__init__()

        self.t_end_star = float(t_end_star)

        if activation == "tanh":
            act = nn.Tanh
        elif activation == "silu":
            act = nn.SiLU
        else:
            raise ValueError("activation must be 'tanh' or 'silu'")

        layers = []
        in_dim = 2

        for i in range(depth):
            layers.append(nn.Linear(in_dim if i == 0 else width, width))
            layers.append(act())

        self.net = nn.Sequential(*layers)
        self.head = nn.Linear(width, 3)

        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, z, t):
        z_scaled = 2.0 * z - 1.0
        t_scaled = 2.0 * t / self.t_end_star - 1.0

        x = torch.cat([z_scaled, t_scaled], dim=1)
        y = self.head(self.net(x))

        G_hat = y[:, 0:1]
        h_hat = y[:, 1:2]
        Pi_hat = y[:, 2:3]

        return G_hat, h_hat, Pi_hat

In [ ]:
class SineLayer(nn.Module):
    def __init__(self, in_dim, out_dim, omega0=1.0, first=False):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.omega0 = omega0
        self.first = first
        self.reset_parameters()

    def reset_parameters(self):
        in_dim = self.linear.in_features
        with torch.no_grad():
            if self.first:
                bound = 1.0 / in_dim
            else:
                bound = math.sqrt(6.0 / in_dim) / self.omega0
            self.linear.weight.uniform_(-bound, bound)
            self.linear.bias.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega0 * self.linear(x))


class SirenPINN(nn.Module):
    def __init__(self, width=64, depth=4, omega0_first=30.0, omega0_hidden=1.0):
        super().__init__()
        layers = [SineLayer(2, width, omega0=omega0_first, first=True)]
        for _ in range(depth - 1):
            layers.append(SineLayer(width, width, omega0=omega0_hidden))
        self.net = nn.Sequential(*layers)
        self.head = nn.Linear(width, 4)

    def forward(self, z_star, t_star):
        x = torch.cat([z_star, t_star], dim=1)
        y = self.head(self.net(x))

        rho = F.softplus(y[:, 0:1]) + 1e-8
        G = y[:, 1:2]
        h = y[:, 2:3]
        pi = y[:, 3:4]

        return rho, G, h, pi

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, freqs=(1.0, 2.0, 4.0, 8.0)):
        super().__init__()
        self.register_buffer(
            "freqs",
            torch.tensor(freqs, dtype=torch.float32).reshape(1, 1, -1),
        )

    def forward(self, x):
        # x: [N, 2], already scaled to [-1, 1]
        xb = x.unsqueeze(-1)                         # [N, 2, 1]
        arg = math.pi * xb * self.freqs              # [N, 2, F]

        sin = torch.sin(arg).flatten(start_dim=1)
        cos = torch.cos(arg).flatten(start_dim=1)

        return torch.cat([x, sin, cos], dim=1)


class HybridPINN(nn.Module):
    def __init__(
        self,
        t_end_star,
        width_smooth=128,
        width_fourier=64,
        depth_smooth=6,
        depth_fourier=3,
        freqs=(1.0, 2.0, 4.0, 8.0),
    ):
        super().__init__()

        self.t_end_star = float(t_end_star)
        self.fourier = FourierFeatures(freqs=freqs)

        smooth_layers = []
        in_dim = 2

        for i in range(depth_smooth):
            smooth_layers.append(nn.Linear(in_dim if i == 0 else width_smooth, width_smooth))
            smooth_layers.append(nn.Tanh())

        self.smooth_net = nn.Sequential(*smooth_layers)
        self.smooth_head = nn.Linear(width_smooth, 3)

        fourier_dim = 2 + 2 * 2 * len(freqs)

        fourier_layers = []
        for i in range(depth_fourier):
            fourier_layers.append(nn.Linear(fourier_dim if i == 0 else width_fourier, width_fourier))
            fourier_layers.append(nn.Tanh())

        self.fourier_net = nn.Sequential(*fourier_layers)
        self.fourier_head = nn.Linear(width_fourier, 3)

        self.gate_logit = nn.Parameter(torch.full((1, 3), -2.0))

        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def scale_inputs(self, z, t):
        z_scaled = 2.0 * z - 1.0
        t_scaled = 2.0 * t / self.t_end_star - 1.0
        return torch.cat([z_scaled, t_scaled], dim=1)

    def forward(self, z, t):
        x = self.scale_inputs(z, t)

        y_smooth = self.smooth_head(self.smooth_net(x))

        x_fourier = self.fourier(x)
        y_fourier = self.fourier_head(self.fourier_net(x_fourier))

        gate = torch.sigmoid(self.gate_logit)

        y = y_smooth + gate * y_fourier

        G_hat = y[:, 0:1]
        h_hat = y[:, 1:2]
        Pi_hat = y[:, 2:3]

        return G_hat, h_hat, Pi_hat

In [ ]:
def d(y, x):
    return torch.autograd.grad(
        y,
        x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]


def mse(x):
    return torch.mean(x**2)


def normal_pdf(x):
    return torch.exp(-0.5 * x**2) / math.sqrt(2.0 * math.pi)


def normal_cdf(x):
    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))


def delta_inlet(z, alpha):
    a = torch.as_tensor(alpha, dtype=z.dtype, device=z.device)
    denom = a * (normal_cdf(1.0 / a) - normal_cdf(torch.zeros_like(a)))

    return normal_pdf(z / a) / denom


def delta_outlet(z, alpha):
    a = torch.as_tensor(alpha, dtype=z.dtype, device=z.device)
    denom = a * (normal_cdf(torch.zeros_like(a)) - normal_cdf(-1.0 / a))

    return normal_pdf((z - 1.0) / a) / denom

In [ ]:
class Sampler:
    def __init__(self, case, device, dtype=torch.float32, alpha=0.02):
        self.case = case
        self.device = device
        self.dtype = dtype
        self.alpha = alpha

        self.t_bc = torch.linspace(
            0.0,
            case.t_end_star,
            4096,
            device=device,
            dtype=dtype,
        ).reshape(-1, 1)

        z_in = torch.linspace(0.0, 3.0 * alpha, 512, device=device, dtype=dtype).reshape(-1, 1)
        z_mid = torch.linspace(3.0 * alpha, 1.0 - 3.0 * alpha, 1024, device=device, dtype=dtype).reshape(-1, 1)
        z_out = torch.linspace(1.0 - 3.0 * alpha, 1.0, 512, device=device, dtype=dtype).reshape(-1, 1)

        self.z_ic = torch.cat([z_in, z_mid, z_out], dim=0)

    def uniform(self, n, a, b):
        return a + (b - a) * torch.rand(
            n,
            1,
            device=self.device,
            dtype=self.dtype,
        )

    def time(self, n):
        n0 = int(0.10 * n)
        nk = int(0.20 * n)
        nf = n - n0 - nk

        t_full = self.uniform(nf, 0.0, self.case.t_end_star)

        t_start = self.uniform(
            n0,
            0.0,
            min(0.05 * self.case.t_end_star, self.case.t_end_star),
        )

        dt = min(0.5 * self.case.w0 / self.case.L, 0.05 * self.case.t_end_star)
        a = max(0.0, self.case.t_kink_star - dt)
        b = min(self.case.t_end_star, self.case.t_kink_star + dt)

        t_kink = self.uniform(nk, a, b)

        return torch.cat([t_full, t_start, t_kink], dim=0)

    def interior(self, n_in=2048, n_mid=4096, n_out=2048):
        a = self.alpha

        z_in = self.uniform(n_in, 0.0, 3.0 * a)
        z_mid = self.uniform(n_mid, 3.0 * a, 1.0 - 3.0 * a)
        z_out = self.uniform(n_out, 1.0 - 3.0 * a, 1.0)

        return {
            "in": (z_in.requires_grad_(True), self.time(n_in).requires_grad_(True)),
            "mid": (z_mid.requires_grad_(True), self.time(n_mid).requires_grad_(True)),
            "out": (z_out.requires_grad_(True), self.time(n_out).requires_grad_(True)),
        }

    def bc(self):
        t = self.t_bc

        return {
            "in": (torch.zeros_like(t), t),
            "out": (torch.ones_like(t), t),
        }

    def ic(self):
        z = self.z_ic
        t = torch.zeros_like(z)

        return z, t


    # ---- causal training grid (Wang, Sankaran, Perdikaris 2024) ----------
    def causal_time_grid(self, n_t, t_lo=0.0, t_hi=None, jitter=True):
        # Sorted time grid t_1 < ... < t_Nt over (t_lo, t_hi]. linspace cells with
        # intra-cell jitter keep strict ordering (needed for the causal cumulative
        # sum). The left edge t_lo is the IC anchor (handled in causal_pde_loss).
        te = self.case.t_end_star if t_hi is None else t_hi
        edges = torch.linspace(t_lo, te, n_t + 1, device=self.device, dtype=self.dtype)
        left = edges[:-1]
        width = float(edges[1] - edges[0])
        if jitter:
            u = torch.rand(n_t, device=self.device, dtype=self.dtype)
        else:
            u = torch.full((n_t,), 0.5, device=self.device, dtype=self.dtype)
        t = left + u * width
        return t.reshape(-1, 1)

    def causal_space(self, n_t, n_z):
        # For each of n_t times, n_z spatial points spanning [0,1] with the
        # in/mid/out zone density of interior(). Returns (n_t, n_z): row i holds
        # the z's for time i, so a row-major flatten groups points by time.
        a = self.alpha
        n_in = max(1, round(0.25 * n_z))
        n_out = max(1, round(0.25 * n_z))
        n_mid = n_z - n_in - n_out

        def block(n, lo, hi):
            return lo + (hi - lo) * torch.rand(
                n_t, n, device=self.device, dtype=self.dtype,
            )

        z_in = block(n_in, 0.0, 3.0 * a)
        z_mid = block(n_mid, 3.0 * a, 1.0 - 3.0 * a)
        z_out = block(n_out, 1.0 - 3.0 * a, 1.0)
        return torch.cat([z_in, z_mid, z_out], dim=1)


In [ ]:
@dataclass(frozen=True)
class LossWeights:
    # per-equation weights INSIDE the per-time residual L_r(t_i)
    mass: float = 1.0
    mom: float = 1.0
    energy: float = 1.0

    # causal anchors (Wang+2024): IC is the t_0 segment, BC folded per-time so it
    # is gated by the same causal front as the PDE residual.
    causal_ic: float = 1000.0  # lambda_ic, matches CausalPINNs (loss =
                               # mean(W*L_t) + lambda_ic*L_ic, IC full weight).
                               # L_ic also enters every w_i's exponent, so driving
                               # IC -> ~1e-5 is what OPENS the causal gate.
    causal_bc: float = 30.0

    # causal time / space resolution
    Nt: int = 64              # number of sorted time segments
    Nz_pde: int = 96          # spatial points per time for the PDE residual
    Nz_keeper: int = 193      # spatial points per time for the integral keepers

    # integral keepers -- now causally weighted by the SAME w_i (no tail
    # concentration). They ride on the front but do not define it.
    mom_integral: float = 20.0
    energy_integral: float = 20.0
    mass_integral: float = 20.0

    # legacy fields kept so pde_loss / the diagnostic+ablation cells still
    # construct. Not used by the causal path.
    ic: float = 20.0
    bc: float = 20.0
    mom_in: float = 1.0
    mom_mid: float = 1.0
    mom_out: float = 1.0


class Physics:
    def __init__(self, model, eos, mu_spline, case, forcing, alpha=0.02):
        self.model = model
        self.eos = eos
        self.mu_spline = mu_spline
        self.case = case
        self.forcing = forcing
        self.alpha = alpha

    def fields(self, z, t):
        G_hat, h_hat, Pi_hat = self.model(z, t)

        G = self.case.G0_star * G_hat
        h = self.case.nspc * h_hat
        pi = self.case.dpi_star * Pi_hat

        return G, h, pi, G_hat, h_hat, Pi_hat

    def eos25(self, h_star):
        h_abs = self.case.h_pc + (self.case.Cp_pc / self.case.beta_pc) * h_star
        p_abs = torch.full_like(h_abs, self.case.p_ref)

        raw = self.eos.eval_torch(p_abs, h_abs)

        rho_value = raw["rho"] / self.case.rho_pc

        chi_h = (
            self.case.Cp_pc
            / (self.case.beta_pc * self.case.rho_pc)
            * raw["rho_h"]
        )

        rho_star = rho_value + chi_h * (h_star - h_star.detach())

        return rho_star, chi_h

    def NQ_prime(self, t):
        tau = t * self.case.L / self.case.w0
        Q = self.forcing.power_torch(tau)

        return (
            self.case.beta_pc
            / self.case.Cp_pc
            * Q
            / (self.case.rho_pc * self.case.w0 * self.case.A)
        )
    
    def distributed_friction(self, G, h, rho):
        h_abs = self.case.h_pc + (self.case.Cp_pc / self.case.beta_pc) * h

        mu = self.mu_spline.eval_torch(h_abs)["mu"]

        G_abs_dim = self.case.rho_pc * self.case.w0 * smooth_abs(G, eps=1e-6)

        Re = G_abs_dim * self.case.Dh / mu

        lambda_darcy = darcy_lambda_torch(
            Re=Re,
            roughness=self.case.roughness,
            Dh=self.case.Dh,
            model=self.case.friction_model,
        )

        Lambda_dyn = lambda_darcy * self.case.L / (2.0 * self.case.Dh)

        signed_loss_flux = G * smooth_abs(G, eps=1e-6) / rho

        return Lambda_dyn, signed_loss_flux, Re, lambda_darcy

    def residuals_raw(self, z, t, include_local=True):
        G, h, pi, _, _, _ = self.fields(z, t)

        G_t = d(G, t)
        G_z = d(G, z)

        h_t = d(h, t)
        h_z = d(h, z)

        pi_z = d(pi, z)

        rho, chi_h = self.eos25(h)

        rho_t = chi_h * h_t
        rho_z = chi_h * h_z

        mom_flux = G**2 / rho

        mom_flux_z = (
            2.0 * G * G_z / rho
            - G**2 * rho_z / rho**2
        )

        Lambda_dyn, signed_loss_flux, Re, lambda_darcy = self.distributed_friction(
            G=G,
            h=h,
            rho=rho,
        )

        delta0 = delta_inlet(z, self.alpha)
        delta1 = delta_outlet(z, self.alpha)

        distributed_loss = Lambda_dyn * signed_loss_flux

        # Local inlet/outlet losses are physically pressure JUMPS at the faces,
        # NOT a distributed interior body force. The Gaussian-smeared delta puts
        # a near-singular spike (G|G|/rho_out blows up as rho_out -> 0.4 when hot)
        # into the POINTWISE residual that a smooth tanh net cannot resolve -> a
        # high outlet-zone residual that drives the optimizer to the cold basin.
        # We therefore EXCLUDE the local term from the pointwise R_mom and enforce
        # it ONLY in the integral momentum balance (integral_mom_loss), where the
        # delta integrates to a finite jump 1/2 k G|G|/rho (no spike). The rho is
        # still anchored at the end face. include_local defaults True so the
        # method stays general; residuals_scaled calls it with include_local=False.
        if include_local:
            z0 = torch.zeros_like(t)
            z1 = torch.ones_like(t)
            _, h_in_ep, _, _, _, _ = self.fields(z0, t)
            _, h_out_ep, _, _, _, _ = self.fields(z1, t)
            rho_in_ep, _ = self.eos25(h_in_ep)
            rho_out_ep, _ = self.eos25(h_out_ep)

            loss_flux_num = G * smooth_abs(G, eps=1e-6)  # numerator G|G|

            local_loss = (
                0.5 * self.case.kin * delta0 * (loss_flux_num / rho_in_ep)
                + 0.5 * self.case.kout * delta1 * (loss_flux_num / rho_out_ep)
            )
        else:
            local_loss = torch.zeros_like(G)

        R_mass = rho_t + G_z

        R_mom = (
            G_t
            + mom_flux_z
            + pi_z
            + rho / self.case.Fr
            + distributed_loss
            + local_loss
        )

        R_energy = rho * h_t + G * h_z - self.NQ_prime(t) + h * R_mass

        return R_mass, R_mom, R_energy

    def residuals_scaled(self, z, t, zone):
        # Pointwise momentum WITHOUT the delta-smeared local losses (those are a
        # face jump, enforced in integral_mom_loss). The interior R_mom is now
        # smooth in every zone, so all zones share the mid scale (dpi_star). The
        # old per-zone mom_delta_scale (~1096) only existed to tame the removed
        # spike; keeping it here would under-weight the in/out zones ~12x.
        R_mass, R_mom, R_energy = self.residuals_raw(z, t, include_local=False)

        R_mass = R_mass / self.case.mass_scale
        R_energy = R_energy / self.case.energy_scale
        R_mom = R_mom / self.case.mom_mid_scale  # zone kept for signature compat

        return R_mass, R_mom, R_energy

    def pde_loss(self, points, weights):
        mass_losses: dict = {}
        energy_losses: dict = {}
        mom_losses: dict = {}

        for zone in ["in", "mid", "out"]:
            z, t = points[zone]
            R_mass, R_mom, R_energy = self.residuals_scaled(z, t, zone)

            mass_losses[zone] = mse(R_mass)
            energy_losses[zone] = mse(R_energy)
            mom_losses[zone] = mse(R_mom)

        L_mass = (
            mass_losses["in"]
            + mass_losses["mid"]
            + mass_losses["out"]
        ) / 3.0

        L_energy = (
            energy_losses["in"]
            + energy_losses["mid"]
            + energy_losses["out"]
        ) / 3.0

        L_mom = (
            weights.mom_in * mom_losses["in"]
            + weights.mom_mid * mom_losses["mid"]
            + weights.mom_out * mom_losses["out"]
        )

        return {
            "mass": L_mass,
            "mass_in": mass_losses["in"],
            "mass_mid": mass_losses["mid"],
            "mass_out": mass_losses["out"],
            "mom": L_mom,
            "mom_in": mom_losses["in"],
            "mom_mid": mom_losses["mid"],
            "mom_out": mom_losses["out"],
            "energy": L_energy,
            "energy_in": energy_losses["in"],
            "energy_mid": energy_losses["mid"],
            "energy_out": energy_losses["out"],
        }

    def ic_loss(self, points):
        z, t = points
        G, h, _, _, _, _ = self.fields(z, t)

        R_G = G / self.case.G0_star - 1.0
        R_h = h / self.case.nspc + 1.0

        return {
            "ic_G": mse(R_G),
            "ic_h": mse(R_h),
        }

    def bc_loss(self, points):
        z_in, t_in = points["in"]
        z_out, t_out = points["out"]

        _, h_in, pi_in, _, _, _ = self.fields(z_in, t_in)
        _, _, pi_out, _, _, _ = self.fields(z_out, t_out)

        R_h_in = h_in / self.case.nspc + 1.0

        # Imposed pressure drop (Ambrosini 2010; Churkin TEMPA-SC):
        # Delta p across the channel is fixed, the flow rate is free to oscillate.
        # One constraint on the DIFFERENCE pi*(0,t) - pi*(1,t) = Delta pi*,
        # plus a gauge anchor pi*(1,t) = 0 (p_out = 25 MPa reference).
        R_dp = (pi_in - pi_out) / self.case.dpi_star - 1.0
        R_pi_out = pi_out / self.case.dpi_star

        return {
            "bc_h_in": mse(R_h_in),
            "bc_dp": mse(R_dp),
            "bc_pi_out": mse(R_pi_out),
        }

    # =====================================================================
    # Causal training (Wang, Sankaran, Perdikaris, CMAME 2024; arXiv:2203.07404)
    # =====================================================================
    def ic_loss_target(self, z, t_at, G_target, h_target):
        # IC anchor against a frozen target state (G_target, h_target)(z) at time
        # t_at -- time-marching: window k>=1 pins its left edge to the previous
        # window's end-state. Same normalization as the physical ic_loss.
        t = torch.full_like(z, float(t_at))
        G, h, _, _, _, _ = self.fields(z, t)
        R_G = (G - G_target) / self.case.G0_star
        R_h = (h - h_target) / self.case.nspc
        return {"ic_G": mse(R_G), "ic_h": mse(R_h)}

    @torch.no_grad()
    def snapshot_state(self, z, t_at):
        # Detached (G, h) profiles at time t_at from THIS model -- becomes the IC
        # target for the next marching window.
        t = torch.full_like(z, float(t_at))
        G, h, _, _, _, _ = self.fields(z, t)
        return G.detach(), h.detach()

    def _per_time_residual(self, sampler, weights, t_grid, ic_spec=None):
        # Per-time residual L_r(t_i) = PDE (mass/mom/energy) + folded BC, computed
        # in ONE vectorized autograd pass over a (Nt, Nz) collocation grid, plus
        # the IC anchor L_0 = lambda_ic * L_ic (the t_0 segment, Eq. 3.4).
        # Returns (Lr, L0, parts): Lr is (Nt,), L0 scalar, parts for logging.
        Nt = t_grid.shape[0]

        # --- PDE residual on (Nt, Nz) points (random z per time) -------------
        z_mat = sampler.causal_space(Nt, weights.Nz_pde)     # (Nt, Nz_a)
        Nz = z_mat.shape[1]
        Z = z_mat.reshape(-1, 1).clone().requires_grad_(True)
        T = t_grid.reshape(-1, 1).repeat_interleave(Nz, dim=0).clone().requires_grad_(True)

        R_mass, R_mom, R_energy = self.residuals_scaled(Z, T, "mid")
        R_mass = R_mass.reshape(Nt, Nz)
        R_mom = R_mom.reshape(Nt, Nz)
        R_energy = R_energy.reshape(Nt, Nz)

        Lr_mass = (R_mass ** 2).mean(dim=1)                  # (Nt,)
        Lr_mom = (R_mom ** 2).mean(dim=1)
        Lr_energy = (R_energy ** 2).mean(dim=1)
        Lr_pde = (
            weights.mass * Lr_mass
            + weights.mom * Lr_mom
            + weights.energy * Lr_energy
        )

        # --- BC per-time (imposed Delta-p), one point per face per time ------
        z0 = torch.zeros_like(t_grid)
        z1 = torch.ones_like(t_grid)
        _, h_in, pi_in, _, _, _ = self.fields(z0, t_grid)
        _, _, pi_out, _, _, _ = self.fields(z1, t_grid)
        R_h_in = h_in / self.case.nspc + 1.0
        R_dp = (pi_in - pi_out) / self.case.dpi_star - 1.0
        R_pi_out = pi_out / self.case.dpi_star
        Lr_bc = (R_h_in ** 2 + R_dp ** 2 + R_pi_out ** 2).reshape(-1)   # (Nt,)

        # BC is NOT folded into the causal residual: imposed-Delta-p is a
        # STANDING constraint (must hold at every t). Causally gating it lets the
        # high-eps weights switch off the tail BC (w_i -> 0) -> pi_out drifts and
        # Delta-p overshoots. Returned separately, enforced uniformly below.

        # --- IC anchor (left edge of the window) -----------------------------
        # window 0: physical IC at t=0; window k>=1: seam snapshot at t_lo.
        if ic_spec is None:
            ic = self.ic_loss(sampler.ic())
        else:
            ic = self.ic_loss_target(
                ic_spec["z"], ic_spec["t_at"], ic_spec["G"], ic_spec["h"],
            )
        L_ic = ic["ic_G"] + ic["ic_h"]
        L0 = weights.causal_ic * L_ic

        parts = {
            "pde": Lr_pde.mean().detach(),
            "bc": Lr_bc.mean().detach(),
            "ic": L_ic.detach(),
        }
        return Lr_pde, L0, Lr_bc, parts

    def _keeper_vectors(self, t_grid, n_z=193):
        # SIGNED per-time integral residuals of the three keeper identities on a
        # uniform z-grid -- returns (mom_vec, energy_vec, mass_vec), each (Nt,),
        # with the autograd graph intact. Vectorized: one forward+autograd pass
        # over (Nt, Nz). Single source of truth for both the causal keeper loss
        # and keeper_time_profiles. Mirrors the old integral_*_loss term by term.
        p = next(self.model.parameters())
        device, dtype = p.device, p.dtype
        Nt = t_grid.shape[0]
        z_lin = torch.linspace(0.0, 1.0, n_z, device=device, dtype=dtype)
        dz = float(z_lin[1] - z_lin[0])

        Z = z_lin.reshape(1, n_z).expand(Nt, n_z).reshape(-1, 1).clone().requires_grad_(True)
        T = t_grid.reshape(-1, 1).repeat_interleave(n_z, dim=0).clone().requires_grad_(True)

        G, h, pi, _, _, _ = self.fields(Z, T)
        rho, chi_h = self.eos25(h)

        G_t = d(G, T)
        h_t = d(h, T)
        h_z = d(h, Z)
        G_z = d(G, Z)
        rho_t = chi_h * h_t

        Lambda_dyn, signed_loss_flux, _, _ = self.distributed_friction(
            G=G, h=h, rho=rho,
        )
        loss_flux_num = G * smooth_abs(G, eps=1e-6)
        di = delta_inlet(Z, self.alpha)
        do = delta_outlet(Z, self.alpha)

        def M(x):
            return x.reshape(Nt, n_z)

        flux = M(G ** 2 / rho)
        rho_m = M(rho)
        lfn_m = M(loss_flux_num)
        di_m = M(di)
        do_m = M(do)

        accel = torch.trapezoid(M(G_t), dx=dz, dim=1)            # (Nt,)
        gravity = torch.trapezoid(M(rho / self.case.Fr), dx=dz, dim=1)
        friction = torch.trapezoid(M(Lambda_dyn * signed_loss_flux), dx=dz, dim=1)
        flux_jump = flux[:, -1] - flux[:, 0]
        local_in = torch.trapezoid(
            0.5 * self.case.kin * di_m * lfn_m / rho_m[:, 0:1], dx=dz, dim=1,
        )
        local_out = torch.trapezoid(
            0.5 * self.case.kout * do_m * lfn_m / rho_m[:, -1:], dx=dz, dim=1,
        )
        rhs = accel + flux_jump + gravity + friction + local_in + local_out
        mom_vec = (rhs - self.case.dpi_star) / self.case.dpi_star   # (Nt,)

        R_mass_m = M(rho_t + G_z)
        R_energy_m = M(rho * h_t + G * h_z - self.NQ_prime(T)) + M(h) * R_mass_m
        energy_vec = torch.trapezoid(R_energy_m / self.case.energy_scale, dx=dz, dim=1)
        mass_vec = torch.trapezoid(R_mass_m / self.case.mass_scale, dx=dz, dim=1)

        return mom_vec, energy_vec, mass_vec

    def causal_pde_loss(self, sampler, weights, eps, t_lo=0.0, t_hi=None, ic_spec=None):
        # Total causal loss (Eq. 3.2-3.4) over the window (t_lo, t_hi]. Builds the
        # sorted time grid, per-time residual L_r(t_i), causal weights w_i
        # (stop-gradient), the weighted sum, and the causally weighted keepers.
        t_grid = sampler.causal_time_grid(weights.Nt, t_lo, t_hi)
        Lr, L0, Lr_bc, parts = self._per_time_residual(sampler, weights, t_grid, ic_spec)

        # causal weights from PDE residual + IC anchor ONLY (BC is enforced
        # uniformly below; it must not be gated). seg[0]=IC, seg[1:]=L_r(t_i).
        seg = torch.cat([L0.reshape(1), Lr])                 # (Nt+1,)
        S = torch.cumsum(seg, dim=0) - seg                   # exclusive prefix sum
        w = torch.exp(-eps * S.detach())                     # stop-gradient on w_i
        W = w[1:]                                            # temporal weights (== CausalPINNs W)
        w_min = W.min()
        # CausalPINNs total (KS/regular_KS.py): loss = mean(W*L_t) + L_0.
        # IC anchor at FULL weight (not averaged); residual = mean of weighted chunks.
        L_weighted = L0 + (W * Lr).mean()

        # BC: uniform FULL weight at all times (NOT causally gated)
        L_bc = weights.causal_bc * Lr_bc.mean()

        # causalized keepers: same w_i, same t_grid, no tail concentration
        mom_vec, energy_vec, mass_vec = self._keeper_vectors(t_grid, weights.Nz_keeper)
        w_k = W                                              # same temporal weights as the residual
        L_mom_int = (w_k * mom_vec ** 2).mean()
        L_energy_int = (w_k * energy_vec ** 2).mean()
        L_mass_int = (w_k * mass_vec ** 2).mean()

        total = (
            L_weighted
            + L_bc
            + weights.mom_integral * L_mom_int
            + weights.energy_integral * L_energy_int
            + weights.mass_integral * L_mass_int
        )

        logs = {
            "loss": total,
            "pde": parts["pde"],
            "bc": parts["bc"],
            "ic": parts["ic"],
            "w_min": w_min.detach(),
            "eps": float(eps),
            # raw (unweighted) mean-square keeper drift -- comparable to old logs
            "mom_int": (mom_vec ** 2).mean().detach(),
            "energy_int": (energy_vec ** 2).mean().detach(),
            "mass_int": (mass_vec ** 2).mean().detach(),
        }
        return total, logs

    def keeper_time_profiles(self, n_t=80, n_z=193):
        # SIGNED per-time keeper integrals on a uniform time grid -- for the
        # per-keeper diagnostic and ablation. Thin detached wrapper over
        # _keeper_vectors (single source of truth with the causal loss).
        p = next(self.model.parameters())
        device, dtype = p.device, p.dtype
        te = self.case.t_end_star
        t_grid = torch.linspace(0.0, te, n_t, device=device, dtype=dtype).reshape(-1, 1)
        mom_vec, energy_vec, mass_vec = self._keeper_vectors(t_grid, n_z)
        t_star = [float(x) for x in t_grid.reshape(-1).detach().cpu()]
        t_sec = [x * self.case.L / self.case.w0 for x in t_star]
        return {
            "t_star": t_star,
            "t_sec": t_sec,
            "mom": [float(x) for x in mom_vec.detach().cpu()],
            "energy": [float(x) for x in energy_vec.detach().cpu()],
            "mass": [float(x) for x in mass_vec.detach().cpu()],
        }

    def causal_weight_profile(self, sampler, weights, eps, n_t=None):
        # Diagnostic: per-segment causal weight w_i and residual L_r(t_i) on a
        # no-jitter grid, for plotting the causal front. Read-only (detached).
        Nt = int(n_t or weights.Nt)
        t_grid = sampler.causal_time_grid(Nt, jitter=False)
        Lr, L0, _, _ = self._per_time_residual(sampler, weights, t_grid)
        seg = torch.cat([L0.reshape(1), Lr])
        S = torch.cumsum(seg, dim=0) - seg
        w = torch.exp(-eps * S)
        t_star = [0.0] + [float(x) for x in t_grid.reshape(-1).detach().cpu()]
        return {
            "t_star": t_star,
            "t_sec": [x * self.case.L / self.case.w0 for x in t_star],
            "Lr": [float(L0.detach())] + [float(x) for x in Lr.detach().cpu()],
            "w": [float(x) for x in w.detach().cpu()],
        }


In [ ]:
class Trainer:
    def __init__(self, model, physics, sampler, weights, lr=1e-4):
        self.model = model
        self.physics = physics
        self.sampler = sampler
        self.weights = weights
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    def step(self, eps, t_lo=0.0, t_hi=None, ic_spec=None):
        self.optimizer.zero_grad(set_to_none=True)

        loss, logs = self.physics.causal_pde_loss(
            self.sampler, self.weights, eps, t_lo, t_hi, ic_spec,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

        self.optimizer.step()

        return {
            key: (float(value.detach().cpu()) if torch.is_tensor(value) else float(value))
            for key, value in logs.items()
        }


class StitchedModel(nn.Module):
    # Routes (z, t) to the window model that owns t, so the marched solution acts
    # as one model over [0, t_end] for all the diagnostic cells. Each submodel is
    # evaluated on all points and masked (differentiable; cheap at diagnostic
    # sizes). Windows partition time as [lo, hi) except the last, which is [lo, hi].
    def __init__(self, windows, t_end_star):
        super().__init__()
        self.submodels = nn.ModuleList([m for (_, _, m) in windows])
        self.bounds = [(float(lo), float(hi)) for (lo, hi, _) in windows]
        self.t_end_star = float(t_end_star)

    def forward(self, z, t):
        G = torch.zeros_like(z)
        H = torch.zeros_like(z)
        P = torch.zeros_like(z)
        n = len(self.bounds)
        for j, ((lo, hi), m) in enumerate(zip(self.bounds, self.submodels)):
            mask = (t >= lo) & (t < hi) if j < n - 1 else (t >= lo) & (t <= hi)
            sel = mask.to(z.dtype)
            g, h, pp = m(z, t)
            G = G + sel * g
            H = H + sel * h
            P = P + sel * pp
        return G, H, P



In [ ]:
class PhysicsCV(Physics):
    """Physics с интегральным (конечно-объёмным) движком PDE-невязок (Zhang+ 2026, 1D-адаптация).
    Заменён ТОЛЬКО _per_time_residual: causal_pde_loss/keepers/BC/IC наследуются."""

    def __init__(self, *args, n_cv=128, w_bnd=0.06, **kw):
        super().__init__(*args, **kw)
        self.n_cv = int(n_cv)
        self.w_bnd = float(w_bnd)

    def _cv_grid(self, device, dtype):
        # граничные CV [0, w_bnd] и [1-w_bnd, 1] (зона локальных потерь), внутри равномерно
        inner = torch.linspace(self.w_bnd, 1.0 - self.w_bnd, self.n_cv - 1, device=device, dtype=dtype)
        faces = torch.cat([torch.zeros(1, device=device, dtype=dtype), inner,
                           torch.ones(1, device=device, dtype=dtype)])
        mids = 0.5 * (faces[1:] + faces[:-1])
        dz = faces[1:] - faces[:-1]
        return faces, mids, dz

    def _per_time_residual(self, sampler, weights, t_grid, ic_spec=None):
        Nt = t_grid.shape[0]
        device, dtype = t_grid.device, t_grid.dtype
        faces, mids, dz = self._cv_grid(device, dtype)
        Nf, Nc = faces.numel(), mids.numel()

        # --- грани: потоки (z-производные не нужны, автоград только по t в центрах) ---
        Zf = faces.reshape(1, Nf).expand(Nt, Nf).reshape(-1, 1)
        Tf = t_grid.reshape(-1, 1).repeat_interleave(Nf, dim=0)
        G_f, h_f, pi_f, _, _, _ = self.fields(Zf, Tf)
        rho_f, _ = self.eos25(h_f)
        M_f = lambda x: x.reshape(Nt, Nf)
        flux_f = M_f(G_f ** 2 / rho_f + pi_f)
        Gh_f = M_f(G_f * h_f)
        Gm_f = M_f(G_f)

        # --- центры: объёмные члены с d/dt автоградом ---
        Zm = mids.reshape(1, Nc).expand(Nt, Nc).reshape(-1, 1)
        Tm = t_grid.reshape(-1, 1).repeat_interleave(Nc, dim=0).clone().requires_grad_(True)
        G_m, h_m, _, _, _, _ = self.fields(Zm, Tm)
        rho_m, chi_m = self.eos25(h_m)
        G_t = d(G_m, Tm)
        h_t = d(h_m, Tm)
        Lambda_dyn, slf, _, _ = self.distributed_friction(G=G_m, h=h_m, rho=rho_m)
        NQ = self.NQ_prime(Tm)
        M_c = lambda x: x.reshape(Nt, Nc)
        dzr = dz.reshape(1, Nc)

        # масса: ∫χ h_t dz + [G]
        R_mass = M_c(chi_m * h_t) * dzr + (Gm_f[:, 1:] - Gm_f[:, :-1])
        # импульс: ∫G_t dz + [G²/ρ+π] + ∫(ρ/Fr + трение) dz + скачки в граничных CV
        R_mom = (M_c(G_t) * dzr
                 + (flux_f[:, 1:] - flux_f[:, :-1])
                 + M_c(rho_m / self.case.Fr + Lambda_dyn * slf) * dzr)
        lfn_f = M_f(G_f * smooth_abs(G_f, eps=1e-6))
        rho_fm = M_f(rho_f)
        jump_in = 0.5 * self.case.kin * lfn_f[:, 0] / rho_fm[:, 0]
        jump_out = 0.5 * self.case.kout * lfn_f[:, -1] / rho_fm[:, -1]
        R_mom = R_mom.clone()
        R_mom[:, 0] = R_mom[:, 0] + jump_in
        R_mom[:, -1] = R_mom[:, -1] + jump_out
        # энергия: ∫(ρ+hχ) h_t dz + [G h] − ∫N_Q' dz
        R_energy = (M_c((rho_m + h_m * chi_m) * h_t) * dzr
                    + (Gh_f[:, 1:] - Gh_f[:, :-1])
                    - M_c(NQ) * dzr)

        # интенсивная нормировка: /Δz и штатные масштабы — пол сравним с точечным AD
        R_mass = R_mass / (dzr * self.case.mass_scale)
        R_mom = R_mom / (dzr * self.case.mom_mid_scale)
        R_energy = R_energy / (dzr * self.case.energy_scale)

        Lr_pde = (weights.mass * (R_mass ** 2).mean(dim=1)
                  + weights.mom * (R_mom ** 2).mean(dim=1)
                  + weights.energy * (R_energy ** 2).mean(dim=1))

        # --- BC per-time (штатный imposed-Δp) — дословно из Physics ---
        z0 = torch.zeros_like(t_grid)
        z1 = torch.ones_like(t_grid)
        _, h_in, pi_in, _, _, _ = self.fields(z0, t_grid)
        _, _, pi_out, _, _, _ = self.fields(z1, t_grid)
        R_h_in = h_in / self.case.nspc + 1.0
        R_dp = (pi_in - pi_out) / self.case.dpi_star - 1.0
        R_pi_out = pi_out / self.case.dpi_star
        Lr_bc = (R_h_in ** 2 + R_dp ** 2 + R_pi_out ** 2).reshape(-1)

        # --- IC-якорь — дословно из Physics ---
        if ic_spec is None:
            ic = self.ic_loss(sampler.ic())
        else:
            ic = self.ic_loss_target(ic_spec["z"], ic_spec["t_at"], ic_spec["G"], ic_spec["h"])
        L_ic = ic["ic_G"] + ic["ic_h"]
        L0 = weights.causal_ic * L_ic

        parts = {"pde": Lr_pde.mean().detach(), "bc": Lr_bc.mean().detach(), "ic": L_ic.detach()}
        return Lr_pde, L0, Lr_bc, parts


In [ ]:
class ZNet(nn.Module):
    """Поле (G,h,π) как функция z на ОДНОМ шаге: RFF по z + жёсткие BC; выход = отклонение от базового профиля (состояние_n)."""
    def __init__(self, m=64, sigma=1.0, width=128, seed=1):
        super().__init__()
        gen = torch.Generator().manual_seed(seed)
        self.register_buffer("k", torch.randn(m, generator=gen) * sigma)
        self.net = nn.Sequential(nn.Linear(2*m, width), nn.Tanh(), nn.Linear(width, width), nn.Tanh(), nn.Linear(width, 3))
        self.base_h = None; self.base_G = None
    def set_base(self, zg, h_n, G_n):
        # линейная интерполяция базового профиля по z (буферы)
        self.zg, self.hb, self.Gb = zg.reshape(-1), h_n.reshape(-1).detach(), G_n.reshape(-1).detach()
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)   # старт = состояние_n
    def _interp(self, v, z):
        zq = z.reshape(-1).clamp(self.zg[0], self.zg[-1])
        i = torch.searchsorted(self.zg, zq.contiguous(), right=True).clamp(1, self.zg.numel()-1)
        w = (zq - self.zg[i-1]) / (self.zg[i] - self.zg[i-1])
        return ((1-w)*v[i-1] + w*v[i]).reshape(-1, 1)
    def forward(self, z, t=None):
        arg = 2*math.pi*z*self.k.reshape(1, -1)
        q = self.net(torch.cat([torch.cos(arg), torch.sin(arg)], 1))
        G = self._interp(self.Gb, z) + case.G0_star * q[:, 0:1]
        h = self._interp(self.hb, z) + case.nspc * z * q[:, 1:2]          # h(0) = -NSPC точно (базовый профиль тоже)
        Pi_hat = (1.0 - z) + z*(1.0 - z) * q[:, 2:3]                       # imposed-dp и gauge точно
        return G/case.G0_star, h/case.nspc, Pi_hat


In [ ]:
import os, time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
torch.manual_seed(0)

physics = PhysicsCV(None, eos, mu_spline, case, forcing, alpha=0.02, n_cv=128, w_bnd=0.06)
faces, mids, dz = physics._cv_grid(device, dtype)
Nf, Nc = faces.numel(), mids.numel()
Zf, Zm, dzc = faces.reshape(-1, 1), mids.reshape(-1, 1), dz.reshape(-1, 1)
W_MASS, W_MOM, W_EN = 2.0, 0.25, 0.5          # штатные веса уравнений

def cv_terms(model, t_star):
    """CV-члены LSFVP в момент t_star на полях model: (ρ, G, ρh на центрах) и правые части F по уравнениям (без ∂/∂t)."""
    physics.model = model
    tf = torch.full((Nf, 1), float(t_star), device=device, dtype=dtype); tm = torch.full((Nc, 1), float(t_star), device=device, dtype=dtype)
    G_f, h_f, pi_f, _, _, _ = physics.fields(Zf, tf); rho_f, _ = physics.eos25(h_f)
    G_m, h_m, _, _, _, _ = physics.fields(Zm, tm); rho_m, _ = physics.eos25(h_m)
    Ld, slf, _, _ = physics.distributed_friction(G=G_m, h=h_m, rho=rho_m); NQ = physics.NQ_prime(tm)
    F_mass = (G_f[1:] - G_f[:-1])
    fluxf = G_f**2/rho_f + pi_f; lfn = G_f * smooth_abs(G_f, eps=1e-6)
    F_mom = (fluxf[1:] - fluxf[:-1]) + (rho_m/case.Fr + Ld*slf)*dzc
    F_mom = F_mom.clone(); F_mom[0] = F_mom[0] + 0.5*case.kin*lfn[0]/rho_f[0]; F_mom[-1] = F_mom[-1] + 0.5*case.kout*lfn[-1]/rho_f[-1]
    F_en = (G_f[1:]*h_f[1:] - G_f[:-1]*h_f[:-1]) - NQ*dzc
    return dict(rho=rho_m, G=G_m, rhoh=rho_m*h_m, h=h_m, F_mass=F_mass, F_mom=F_mom, F_en=F_en, G_in=G_f[0:1], G_out=G_f[-1:])

def step_loss(model, t_n, dt, S_n, theta):
    """Невязка неявного шага: (u_{n+1}-u_n)/dt·Δz + θ·F_{n+1} + (1-θ)·F_n = 0.  θ=1: BE, θ=0.5: CN.  S_n — детач-члены прошлого шага."""
    S1 = cv_terms(model, t_n + dt)
    R_mass = ((S1["rho"] - S_n["rho"])/dt*dzc + theta*S1["F_mass"] + (1-theta)*S_n["F_mass"]) / (dzc*case.mass_scale)
    R_mom  = ((S1["G"]   - S_n["G"])  /dt*dzc + theta*S1["F_mom"]  + (1-theta)*S_n["F_mom"])  / (dzc*case.mom_mid_scale)
    R_en   = ((S1["rhoh"]- S_n["rhoh"])/dt*dzc + theta*S1["F_en"]  + (1-theta)*S_n["F_en"])   / (dzc*case.energy_scale)
    return W_MASS*(R_mass**2).mean() + W_MOM*(R_mom**2).mean() + W_EN*(R_en**2).mean(), S1

def detach_terms(S): return {k: v.detach() for k, v in S.items()}

def take_step(znet, t_n, dt, S_n, theta, iters):
    znet.set_base(mids, S_n["h"], S_n["G"])
    opt = torch.optim.LBFGS(znet.parameters(), max_iter=iters, history_size=30, line_search_fn="strong_wolfe",
                            tolerance_grad=1e-11, tolerance_change=1e-13)
    def closure():
        opt.zero_grad(set_to_none=True); L, _ = step_loss(znet, t_n, dt, S_n, theta); L.backward(); return L
    opt.step(closure)
    with torch.enable_grad():
        L, S1 = step_loss(znet, t_n, dt, S_n, theta)
    return float(L), detach_terms(S1)

# --- расписание шагов: крупно на гладком участке, мелко в зоне DWO ---
T_FINE_S = float(os.environ.get("V11_TFINE", "1400"))     # s, начало мелкого шага
DT_COARSE, DT_FINE = float(os.environ.get("V11_DTC", "1.0")), float(os.environ.get("V11_DTF", "0.1"))   # t*
THETA = float(os.environ.get("V11_THETA", "1.0"))          # 1.0 = BE, 0.5 = CN
ITERS = int(os.environ.get("V11_ITERS", "60"))
t_fine = T_FINE_S * case.w0 / case.L
def schedule():
    t, out = 0.0, []
    while t < case.t_end_star - 1e-9:
        dt = DT_COARSE if t < t_fine else DT_FINE
        dt = min(dt, case.t_end_star - t); out.append((t, dt)); t += dt
    return out
SCHED = schedule()
CKPT = Path("v11_bemarch_ckpt.pt"); RESUME = True
print(f"схема θ={THETA} ({'BE' if THETA==1 else 'CN'}), шагов {len(SCHED)} (coarse {sum(1 for _,d in SCHED if d==DT_COARSE)} × {DT_COARSE} t*, fine × {DT_FINE} t* от {T_FINE_S:.0f} s), L-BFGS ≤{ITERS}/шаг, {device}")


In [ ]:
# --- начальное состояние: штатный холодный IC good_enough (G = G0*, h = -NSPC при t=0) ---
class _IC(nn.Module):
    def forward(self, z, t): return torch.ones_like(z), -torch.ones_like(z), (1.0 - z)
znet = ZNet().to(device=device, dtype=dtype)
state = {"i": 0, "t": [], "Gin": [], "Gout": [], "loss": [], "hsnap": [], "tsnap": []}
S_n = detach_terms(cv_terms(_IC().to(device), 0.0))
if RESUME and CKPT.exists():
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    state, S_n = ck["state"], {k: v.to(device) for k, v in ck["S_n"].items()}
    print(f"resume: {state['i']} шагов пройдено ({state['t'][-1]*case.L/case.w0:.1f} s)")

k_flow, k_time = case.mdot0/case.G0_star, case.L/case.w0
t0_wall = time.time(); PRINT_EVERY = 100; SNAP_EVERY = 25; SAVE_EVERY = 100
for i in range(state["i"], len(SCHED)):
    t_n, dt = SCHED[i]
    L, S_n = take_step(znet, t_n, dt, S_n, THETA, ITERS)
    state["t"].append(t_n + dt); state["Gin"].append(float(S_n["G_in"]) * k_flow); state["Gout"].append(float(S_n["G_out"]) * k_flow); state["loss"].append(L)
    if i % SNAP_EVERY == 0: state["hsnap"].append(S_n["h"].reshape(-1).cpu().numpy()); state["tsnap"].append(t_n + dt)
    state["i"] = i + 1
    if (i + 1) % PRINT_EVERY == 0 or i + 1 == len(SCHED):
        print(f"шаг {i+1:5d}/{len(SCHED)}  t={(t_n+dt)*k_time:7.1f} s  G_in={state['Gin'][-1]:.5f} kg/s  loss={L:.2e}  ({time.time()-t0_wall:.0f} s)")
    if (i + 1) % SAVE_EVERY == 0 or i + 1 == len(SCHED):
        torch.save({"state": state, "S_n": {k: v.cpu() for k, v in S_n.items()}}, CKPT)
print(f"марш завершён: {state['i']} шагов, средний лосс шага {np.mean(state['loss']):.2e}, макс {np.max(state['loss']):.2e}")


In [ ]:
t_sec = np.array(state["t"]) * k_time; Gin = np.array(state["Gin"]); Gout = np.array(state["Gout"])
ref = np.load("fv_reference_curve.npz")
m = ref["t_sec"] <= 1500.0
env = float(np.mean(np.abs(np.interp(ref["t_sec"][m], t_sec, Gin) - ref["Gin_kgs"][m])) / np.mean(np.abs(ref["Gin_kgs"][m])))
print(f"огибающая vs FV (t<1500 s): {100*env:.2f}%  (ворота 5%: {'PASS' if env <= 0.05 else 'FAIL'})")
print(f"невязка шага: средняя {np.mean(state['loss']):.2e} (ворота 1e-3: {'PASS' if np.mean(state['loss']) < 1e-3 else 'FAIL'})")
w = 20
p2p = np.array([np.ptp(Gin[max(0, j-w):j+w]) for j in range(len(Gin))])
qs = (t_sec > 1400) & (t_sec < 1560); base_p2p = p2p[qs].mean()
hit = np.where((p2p > 10*base_p2p) & (t_sec > 1500))[0]; onset = float(t_sec[hit[0]]) if len(hit) else float("nan")
mt = t_sec > 1650
if mt.sum() > 50:
    s = Gin[mt] - Gin[mt].mean(); dt_s = float(np.mean(np.diff(t_sec[mt])))
    fr = np.fft.rfftfreq(len(s), d=dt_s); sp = np.abs(np.fft.rfft(s)); period = 1.0/fr[int(np.argmax(sp[1:]))+1]
else:
    period = float("nan")
print(f"DWO: onset {onset:.0f} s (1626±15%: {'PASS' if abs(onset-1626)/1626 <= 0.15 else 'FAIL'}), период {period:.2f} s (5.93±15%: {'PASS' if abs(period-5.93)/5.93 <= 0.15 else 'FAIL'}), размах хвоста {np.ptp(Gin[mt]) if mt.any() else float('nan'):.4f} kg/s (FV {np.ptp(ref['Gin_kgs'][ref['t_sec']>1650]):.4f})")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(forcing.time, forcing.flow_in, color="k", lw=1.1, alpha=0.55, label="baseline flow-in")
ax[0].plot(forcing.time, forcing.flow_out, color="tab:green", lw=1.1, alpha=0.7, label="baseline flow-out")
ax[0].plot(ref["t_sec"], ref["Gin_kgs"], color="tab:orange", lw=1.0, alpha=0.9, label="FV flow-in")
ax[0].plot(t_sec, Gin, color="tab:blue", lw=1.3, label="v11 (задача Коши) flow-in")
ax[0].plot(t_sec, Gout, "--", color="tab:red", lw=1.0, label="v11 flow-out")
ax[0].set_xlabel("time [s]"); ax[0].set_ylabel("mass flow [kg/s]"); ax[0].set_title("mass flow: v11 BE/CN-марш vs FV vs baseline"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
H = np.array(state["hsnap"]).T
im = ax[1].imshow(H, aspect="auto", origin="lower", extent=[state["tsnap"][0]*k_time, state["tsnap"][-1]*k_time, 0, 1])
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("z*"); ax[1].set_title("enthalpy h*(z,t), v11"); fig.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
mm = ref["t_sec"] > 1480
ax[0].plot(ref["t_sec"][mm], ref["Gin_kgs"][mm], color="tab:orange", lw=0.9, label="FV"); ax[0].plot(t_sec[t_sec > 1480], Gin[t_sec > 1480], color="tab:blue", lw=1.1, label="v11")
ax[0].axvline(1626, ls=":", color="k", lw=0.8); ax[0].set_xlabel("time [s]"); ax[0].set_ylabel("G_in [kg/s]"); ax[0].set_title("зона onset/DWO"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].semilogy(t_sec, state["loss"], lw=0.7); ax[1].axhline(1e-3, ls=":", color="gray"); ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("невязка шага"); ax[1].set_title("лосс по шагам"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
